# Patient-Independent EEG Seizure Classification
## Experiment 1: 18-Patient Fixed-Budget LOSO Representation Comparison

## 1. Introduction

Automated seizure detection from scalp EEG is challenging because their patterns can vary substantially across patients. As a result, models that perform well on EEG segments from patients seen during training may not generalize as well to completely unseen patients. This experiment investigates how two EEG representations, raw multichannel waveforms and spectral bandpower features, behave under independent evaluation.

## 2. Data and Preprocessing

### 2.1 Runtime and Dependencies

- **MNE** — loads EDF EEG recordings and applies signal processing.
- **NumPy / Pandas** — handle EEG arrays and patient-level results.
- **PyTorch** — defines and trains the neural-network models.
- **TensorDataset / DataLoader** — batch EEG samples during training and evaluation.
- **scikit-learn** — computes AUROC and AUPRC.
- **Path / re / random / gc** — manage file paths, parse seizure annotations, control reproducibility, and release memory.
- **CUDA** — used automatically when available to accelerate PyTorch computation.

In [ ]:
%pip -q install mne

from google.colab import drive
drive.mount("/content/drive")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 76.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
Mounted at /content/drive


In [ ]:
from pathlib import Path
import re
import random
import gc

import numpy as np
import pandas as pd
import mne
import torch
import torch.nn as nn

from scipy.signal import welch
from scipy.integrate import trapezoid
from scipy.stats import wilcoxon
from itertools import product

from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score

### 2.2 Dataset Paths and Development Cohort

- `DATA_ROOT` points to the downloaded CHB-MIT recordings.
- `LOSO_DATA_ROOT` stores processed patient-level arrays used by the LOSO experiments.
- The development cohort contains **18 patients**, with two selected EDF recordings per patient.
- `chb15`, `chb16`, and `chb18` are excluded from this cohort for later held-out evaluation.

In [ ]:
DATA_ROOT = Path(
    "/content/drive/MyDrive/chbmit_subset"
)

LOSO_DATA_ROOT = Path(
    "/content/drive/MyDrive/chbmit-seizure-detection/loso_subject_data"
)

LOSO_DATA_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

SUBJECT_FILES = {
    "chb01": ["chb01_03.edf", "chb01_04.edf"],
    "chb02": ["chb02_16.edf", "chb02_19.edf"],
    "chb03": ["chb03_01.edf", "chb03_02.edf"],
    "chb04": ["chb04_05.edf", "chb04_08.edf"],
    "chb05": ["chb05_06.edf", "chb05_13.edf"],
    "chb06": ["chb06_01.edf", "chb06_04.edf"],
    "chb07": ["chb07_12.edf", "chb07_13.edf"],
    "chb08": ["chb08_02.edf", "chb08_05.edf"],
    "chb09": ["chb09_06.edf", "chb09_08.edf"],
    "chb10": ["chb10_12.edf", "chb10_20.edf"],
    "chb11": ["chb11_82.edf", "chb11_92.edf"],
    "chb13": ["chb13_19.edf", "chb13_21.edf"],
    "chb14": ["chb14_03.edf", "chb14_04.edf"],
    "chb17": ["chb17a_03.edf", "chb17a_04.edf"],
    "chb19": ["chb19_28.edf", "chb19_29.edf"],
    "chb20": ["chb20_12.edf", "chb20_13.edf"],
    "chb22": ["chb22_20.edf", "chb22_25.edf"],
    "chb23": ["chb23_06.edf", "chb23_08.edf"],
}

DEV_SUBJECTS = [
    "chb01", "chb02", "chb03", "chb04",
    "chb05", "chb06", "chb07", "chb08",
    "chb09", "chb10", "chb11", "chb13",
    "chb14", "chb17", "chb19", "chb20",
    "chb22", "chb23",
]

### 2.3 EEG Configuration

- A fixed **18-channel bipolar montage** keeps input dimensions and channel ordering consistent across patients.
- EEG is divided into **4-second non-overlapping windows**.
- Windows within **60 seconds of seizure activity** are excluded from the negative class to avoid ambiguous samples.

In [ ]:
STANDARD_CHANNELS = [
    "FP1-F7",
    "F7-T7",
    "T7-P7",
    "P7-O1",
    "FP1-F3",
    "F3-C3",
    "C3-P3",
    "P3-O1",
    "FP2-F4",
    "F4-C4",
    "C4-P4",
    "P4-O2",
    "FP2-F8",
    "F8-T8",
    "T8-P8-0",
    "P8-O2",
    "FZ-CZ",
    "CZ-PZ",
]

WINDOW_SECONDS = 4
STEP_SECONDS = 4
EXCLUSION_SECONDS = 60

### 2.4 Seizure Annotation and Window Labels

- CHB-MIT summary files provide seizure start and end times for each recording.
- `parse_summary()` extracts these timestamps using regular expressions.
- A 4-second window is labeled **seizure** when at least **2 seconds overlap** an annotated seizure.
- Otherwise, it is labeled **normal** only if it lies at least 60 seconds away from every seizure.
- Ambiguous windows near seizure boundaries are discarded.

In [ ]:
def parse_summary(path):
    text = Path(path).read_text(errors="ignore")

    blocks = re.split(
        r"File Name:\s*",
        text
    )[1:]

    annotations = {}

    for block in blocks:
        fname = block.splitlines()[0].strip()

        starts = [
            int(x)
            for x in re.findall(
                r"Seizure(?: \d+)? Start Time:\s*(\d+)\s*seconds",
                block
            )
        ]

        ends = [
            int(x)
            for x in re.findall(
                r"Seizure(?: \d+)? End Time:\s*(\d+)\s*seconds",
                block
            )
        ]

        annotations[fname] = list(zip(starts, ends))

    return annotations


SEIZURES = {}

for subject in DEV_SUBJECTS:
    summary_path = (
        DATA_ROOT
        / subject
        / f"{subject}-summary.txt"
    )

    if not summary_path.exists():
        raise FileNotFoundError(
            f"Missing summary: {summary_path}"
        )

    SEIZURES[subject] = parse_summary(summary_path)


def get_window_labels(subject, fname):

    seizure_intervals = (
        SEIZURES[subject].get(fname, [])
    )

    raw = mne.io.read_raw_edf(
        DATA_ROOT / subject / fname,
        preload=False,
        verbose=False
    )

    duration_seconds = (
        raw.n_times
        / raw.info["sfreq"]
    )

    windows = []

    for start in np.arange(
        0,
        duration_seconds - WINDOW_SECONDS,
        STEP_SECONDS
    ):

        end = start + WINDOW_SECONDS
        seizure_overlap = 0

        for seizure_start, seizure_end in seizure_intervals:

            overlap = max(
                0,
                min(end, seizure_end)
                - max(start, seizure_start)
            )

            seizure_overlap = max(
                seizure_overlap,
                overlap
            )

        if seizure_overlap >= WINDOW_SECONDS / 2:
            label = 1

        else:
            safely_normal = all(
                end < seizure_start - EXCLUSION_SECONDS
                or
                start > seizure_end + EXCLUSION_SECONDS
                for seizure_start, seizure_end
                in seizure_intervals
            )

            if safely_normal:
                label = 0
            else:
                continue

        windows.append(
            (start, end, label)
        )

    return windows

### 2.5 Setup Validation

- All required EDF files are checked before preprocessing begins.
- Missing recordings stop execution rather than silently producing an incomplete cohort.
- PyTorch selects a CUDA GPU when available and otherwise falls back to CPU.
- The final printout confirms cohort size, montage size, window settings, exclusion interval, and compute device.

In [ ]:
missing = []

for subject in DEV_SUBJECTS:

    for fname in SUBJECT_FILES[subject]:

        path = DATA_ROOT / subject / fname

        if not path.exists():
            missing.append(str(path))


if missing:

    print("MISSING FILES:")

    for path in missing:
        print(path)

    raise FileNotFoundError(
        f"{len(missing)} required EDF file(s) are missing."
    )


device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("====================================")
print("CLEAN LOSO RUNTIME INITIALIZED")
print("====================================")
print("Development patients:", len(DEV_SUBJECTS))
print("Channels:", len(STANDARD_CHANNELS))
print("Window:", WINDOW_SECONDS, "seconds")
print("Step:", STEP_SECONDS, "seconds")
print("Exclusion:", EXCLUSION_SECONDS, "seconds")
print("Device:", device)
print("All required development EDFs found.")

CLEAN LOSO RUNTIME INITIALIZED
Development patients: 18
Channels: 18
Window: 4 seconds
Step: 4 seconds
Exclusion: 60 seconds
Device: cuda
All required development EDFs found.


### 2.6 Raw EEG Window Extraction

- Each EDF recording is loaded with **MNE**, restricted to the fixed 18-channel montage, and filtered from **0.5–50 Hz**.
- At 256 Hz, each 4-second segment contains **1,024 samples per channel**.
- EEG amplitudes are converted from volts to microvolts.
- The previously defined seizure-labeling policy is applied to each recording.
- Windows remain **unstandardized** at this stage so normalization can later be fit using training patients only within each LOSO fold.

In [ ]:
def extract_subject_windows(subject):
    """
    Extract filtered, unstandardized EEG windows for one subject.

    Returns:
        X: [windows, 18 channels, 1024 timepoints]
        y: [windows]
    """

    X_parts = []
    y_parts = []

    for fname in SUBJECT_FILES[subject]:

        print("  Reading:", fname)

        raw = mne.io.read_raw_edf(
            DATA_ROOT / subject / fname,
            preload=True,
            verbose=False
        )

        # Keep only the fixed 18-channel montage
        raw.pick(STANDARD_CHANNELS)

        # Filter the continuous recording before windowing
        raw.filter(
            l_freq=0.5,
            h_freq=50.0,
            verbose=False
        )

        sfreq = int(raw.info["sfreq"])
        assert sfreq == 256

        # Convert volts to microvolts
        data = (
            raw.get_data() * 1e6
        ).astype(np.float32)

        # Apply the predefined seizure-labeling policy
        labeled_windows = get_window_labels(
            subject,
            fname
        )

        X_file = []
        y_file = []

        for start, end, label in labeled_windows:

            start_sample = int(start * sfreq)

            end_sample = (
                start_sample
                + WINDOW_SECONDS * sfreq
            )

            window = data[
                :,
                start_sample:end_sample
            ]

            if window.shape != (
                len(STANDARD_CHANNELS),
                WINDOW_SECONDS * sfreq
            ):
                continue

            # Keep windows unstandardized here.
            # Fold-specific normalization happens later.
            X_file.append(window)
            y_file.append(label)

        X_file = np.stack(
            X_file
        ).astype(np.float32)

        y_file = np.array(
            y_file,
            dtype=np.int64
        )

        print(
            "   windows:", len(y_file),
            "| seizure:",
            int((y_file == 1).sum()),
            "| normal:",
            int((y_file == 0).sum())
        )

        X_parts.append(X_file)
        y_parts.append(y_file)

    X = np.concatenate(
        X_parts,
        axis=0
    )

    y = np.concatenate(
        y_parts,
        axis=0
    )

    return X, y

### 2.7 Save Processed Patient Arrays

- Extracted windows are saved separately for each patient as compressed `.npz` files.
- Keeping one file per patient preserves patient boundaries for later LOSO splitting.
- Existing patient files are reused so the expensive EDF loading and filtering step does not have to run again.
- Each saved file contains the EEG windows (`X`) and labels (`y`).
- Large arrays are deleted after saving to reduce memory use.

In [ ]:
# Run this cell only if the per-patient .npz files do not already exist.
missing_subject_arrays = [
    s for s in DEV_SUBJECTS
    if not (LOSO_DATA_ROOT / f"{s}.npz").exists()
]

if missing_subject_arrays:
    subject_summary = {}
    for subject in missing_subject_arrays:
        print("\n====================")
        print("SUBJECT:", subject)
        print("====================")
        X_subject, y_subject = extract_subject_windows(subject)
        output_path = LOSO_DATA_ROOT / f"{subject}.npz"
        np.savez_compressed(output_path, X=X_subject, y=y_subject)
        subject_summary[subject] = {
            "windows": len(y_subject),
            "seizure": int((y_subject == 1).sum()),
            "normal": int((y_subject == 0).sum()),
        }
        print("SAVED:", output_path)
        del X_subject, y_subject
        gc.collect()
else:
    print("All 18 per-patient raw EEG arrays already exist.")


All 18 per-patient raw EEG arrays already exist.


## 3. Experimental Design

### 3.1 LOSO Setup

- **Leave-one-subject-out (LOSO)** evaluation holds out one patient at a time and trains on the remaining 17.
- `SEED = 42` fixes the random state so model initialization and training behavior are reproducible.
- `BATCH_SIZE = 128` controls how many EEG windows are processed in each training batch.
- Patient arrays are loaded individually so the train/test split can remain strictly patient-disjoint.

In [ ]:
SEED = 42
BATCH_SIZE = 128


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_subject(subject):
    with np.load(
        LOSO_DATA_ROOT / f"{subject}.npz"
    ) as data:

        X = data["X"].astype(np.float32)
        y = data["y"].astype(np.float32)

    return X, y

### 3.2 Fold-Specific Normalization

- EEG amplitude distributions can differ across channels and patients, so each channel is standardized before training.
- The mean and standard deviation are calculated **only from the 17 training patients in each LOSO fold**.
- The held-out patient is transformed using those training-derived statistics rather than contributing to them.
- This prevents preprocessing leakage from the test patient.

In [ ]:
def fit_raw_scaler(subjects):

    channel_sum = np.zeros(
        len(STANDARD_CHANNELS),
        dtype=np.float64
    )

    channel_sq_sum = np.zeros(
        len(STANDARD_CHANNELS),
        dtype=np.float64
    )

    count = 0

    for subject in subjects:

        X, _ = load_subject(subject)

        channel_sum += X.sum(
            axis=(0, 2),
            dtype=np.float64
        )

        channel_sq_sum += np.square(
            X,
            dtype=np.float64
        ).sum(
            axis=(0, 2)
        )

        count += X.shape[0] * X.shape[2]

        del X

    mean = channel_sum / count

    variance = (
        channel_sq_sum / count
        - mean ** 2
    )

    std = np.sqrt(
        np.maximum(
            variance,
            1e-8
        )
    )

    return (
        mean.astype(np.float32),
        std.astype(np.float32)
    )


def build_raw_arrays(subjects, mean, std):

    X_parts = []
    y_parts = []

    for subject in subjects:

        X, y = load_subject(subject)

        X = (
            X - mean[None, :, None]
        ) / (
            std[None, :, None] + 1e-6
        )

        X_parts.append(
            X.astype(np.float32)
        )

        y_parts.append(
            y.astype(np.float32)
        )

    return (
        np.concatenate(X_parts),
        np.concatenate(y_parts)
    )

### 3.3 Raw EEG CNN

- A **1D convolutional neural network (CNN)** is used for raw EEG because convolutional filters can learn local temporal patterns directly from the waveform.
- The network receives an input of shape **18 channels × 1,024 time samples**.
- Three convolutional blocks increase the feature depth from **18 → 32 → 64 → 128** while progressively reducing the temporal dimension.
- **ReLU** introduces nonlinearity, while max pooling reduces temporal resolution and computation.
- `AdaptiveAvgPool1d(1)` summarizes each learned feature map into a single value, producing a fixed 128-dimensional representation.
- **Dropout (0.3)** is applied before the final classifier to reduce overfitting.
- The final linear layer outputs a single **logit** for binary seizure classification.

In [ ]:
class LOSORawCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(

            nn.Conv1d(
                18,
                32,
                kernel_size=7,
                padding=3
            ),
            nn.ReLU(),
            nn.MaxPool1d(4),

            nn.Conv1d(
                32,
                64,
                kernel_size=5,
                padding=2
            ),
            nn.ReLU(),
            nn.MaxPool1d(4),

            nn.Conv1d(
                64,
                128,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),

            nn.AdaptiveAvgPool1d(1)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )

    def forward(self, x):

        x = self.features(x)

        return (
            self.classifier(x)
            .squeeze(1)
        )


print(LOSORawCNN())

LOSORawCNN(
  (features): Sequential(
    (0): Conv1d(18, 32, kernel_size=(7,), stride=(1,), padding=(3,))
    (1): ReLU()
    (2): MaxPool1d(kernel_size=4, stride=4, padding=0, dilation=1, ceil_mode=False)
    (3): Conv1d(32, 64, kernel_size=(5,), stride=(1,), padding=(2,))
    (4): ReLU()
    (5): MaxPool1d(kernel_size=4, stride=4, padding=0, dilation=1, ceil_mode=False)
    (6): Conv1d(64, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (7): ReLU()
    (8): AdaptiveAvgPool1d(output_size=1)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Dropout(p=0.3, inplace=False)
    (2): Linear(in_features=128, out_features=1, bias=True)
  )
)


### 3.4 Training and Inference Utilities

- `TensorDataset` pairs each EEG window with its binary label, while `DataLoader` divides the data into mini-batches of **128 windows**.
- Training data are shuffled each epoch; held-out patient data are not shuffled because ordering is irrelevant during evaluation.
- `train_one_epoch()` performs the standard PyTorch training loop: forward pass, loss calculation, backpropagation, and optimizer update.
- During evaluation, `torch.no_grad()` disables gradient computation to reduce memory use.
- The model outputs logits, which are converted to seizure probabilities using the **sigmoid** function.

In [ ]:
def make_loader(
    X,
    y,
    shuffle
):

    dataset = TensorDataset(
        torch.from_numpy(X),
        torch.from_numpy(y)
    )

    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle
    )


def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion
):

    model.train()

    losses = []

    for X_batch, y_batch in loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        logits = model(X_batch)

        loss = criterion(
            logits,
            y_batch
        )

        loss.backward()
        optimizer.step()

        losses.append(
            loss.item()
        )

    return np.mean(losses)


def predict_probabilities(
    model,
    loader
):

    model.eval()

    labels = []
    probabilities = []

    with torch.no_grad():

        for X_batch, y_batch in loader:

            X_batch = X_batch.to(device)

            logits = model(X_batch)

            probs = torch.sigmoid(
                logits
            )

            probabilities.extend(
                probs.cpu().numpy()
            )

            labels.extend(
                y_batch.numpy()
            )

    return (
        np.array(labels),
        np.array(probabilities)
    )

### 3.5 One-Epoch Raw EEG LOSO Baseline

- Each LOSO fold holds out **one patient** and trains on the remaining **17 patients**.
- Normalization statistics are fit only on the 17 training patients and then applied to the held-out patient.
- Class imbalance is handled with `BCEWithLogitsLoss` using `pos_weight = n_negative / n_positive`, computed from the training fold only.
- **AdamW** is used for optimization with a learning rate of `1e-3` and weight decay of `1e-4`.
- The model is trained for exactly **one epoch** in this baseline experiment.
- Performance on the held-out patient is measured using **AUROC** and **AUPRC**.
- Because seizure prevalence is very low, AUPRC is especially useful for assessing precision-recall behavior.

In [ ]:
FIXED_EPOCHS = 1


def run_fixed_raw_fold(outer_subject):

    print("\n================================")
    print("OUTER PATIENT:", outer_subject)
    print("================================")

    train_subjects = [
        subject
        for subject in DEV_SUBJECTS
        if subject != outer_subject
    ]

    set_seed(SEED)

    # Fit normalization only on training patients
    mean, std = fit_raw_scaler(
        train_subjects
    )

    X_train, y_train = build_raw_arrays(
        train_subjects,
        mean,
        std
    )

    X_outer, y_outer = build_raw_arrays(
        [outer_subject],
        mean,
        std
    )

    train_loader = make_loader(
        X_train,
        y_train,
        shuffle=True
    )

    outer_loader = make_loader(
        X_outer,
        y_outer,
        shuffle=False
    )

    # Compute class weight from training patients only
    n_positive = (y_train == 1).sum()
    n_negative = (y_train == 0).sum()

    pos_weight = n_negative / n_positive

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(
            [pos_weight],
            dtype=torch.float32,
            device=device
        )
    )

    model = LOSORawCNN().to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-4
    )

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion
    )

    outer_labels, outer_probs = predict_probabilities(
        model,
        outer_loader
    )

    auroc = roc_auc_score(
        outer_labels,
        outer_probs
    )

    auprc = average_precision_score(
        outer_labels,
        outer_probs
    )

    prevalence = outer_labels.mean()

    result = {
        "subject": outer_subject,
        "epochs": FIXED_EPOCHS,
        "train_loss": train_loss,
        "positives": int(outer_labels.sum()),
        "windows": len(outer_labels),
        "prevalence": prevalence,
        "auroc": auroc,
        "auprc": auprc
    }

    print(f"Train loss: {train_loss:.4f}")
    print(f"OUTER AUROC: {auroc:.4f}")
    print(f"OUTER AUPRC: {auprc:.4f}")
    print(f"Positive prevalence: {prevalence:.4f}")

    del (
        X_train,
        y_train,
        X_outer,
        y_outer,
        train_loader,
        outer_loader,
        model
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result

#### Running or Loading the LOSO Results

The full 18-fold LOSO experiment is expensive to recompute. Saved results are loaded by default, while `RERUN_RAW_LOSO = True` can be used to intentionally rerun all folds.

In [ ]:
RAW_RESULTS_PATH = (
    "/content/drive/MyDrive/chbmit-seizure-detection/"
    "fixed_raw_loso_18patients.csv"
)

RERUN_RAW_LOSO = False

if RERUN_RAW_LOSO:

    fixed_raw_results = []

    for subject in DEV_SUBJECTS:
        fixed_raw_results.append(
            run_fixed_raw_fold(subject)
        )

    fixed_raw_df = pd.DataFrame(
        fixed_raw_results
    )

    fixed_raw_df.to_csv(
        RAW_RESULTS_PATH,
        index=False
    )

else:

    fixed_raw_df = pd.read_csv(
        RAW_RESULTS_PATH
    )


display(fixed_raw_df)

print("\nRAW LOSO SUMMARY")
print(
    "Median AUROC:",
    fixed_raw_df["auroc"].median()
)
print(
    "Median AUPRC:",
    fixed_raw_df["auprc"].median()
)

,subject,epochs,train_loss,positives,windows,prevalence,auroc,auprc
0,chb01,1,0.733367,17,1735,0.009798,0.996645,0.946488
1,chb02,1,0.719619,24,1077,0.022284,0.954614,0.248967
2,chb03,1,0.737561,30,1736,0.017281,0.988550,0.645840
3,chb04,1,0.786338,40,5919,0.006758,0.984508,0.251425
4,chb05,1,0.748156,57,1736,0.032834,0.903848,0.804108
5,chb06,1,0.727837,22,6766,0.003252,0.741433,0.009442
6,chb07,1,0.776598,46,4468,0.010295,0.995993,0.853418
7,chb08,1,0.760168,91,1736,0.052419,0.880818,0.582970
8,chb09,1,0.716202,54,7105,0.007600,0.997269,0.944946
9,chb10,1,0.739305,27,3539,0.007629,0.980933,0.312493



RAW LOSO SUMMARY
Median AUROC: 0.9811823351418171
Median AUPRC: 0.6138164852148251


## 4. Spectral Representation

### 4.1 Welch Bandpower Features

- The second representation compresses each raw EEG window into frequency-domain features rather than using the waveform directly.
- **Welch's method** estimates the power spectral density (PSD) of each channel using 1-second segments (`nperseg = 256` at 256 Hz).
- Power is integrated within five conventional EEG frequency bands: **delta (1–4 Hz), theta (4–8 Hz), alpha (8–12 Hz), beta (12–30 Hz), and gamma (30–50 Hz)**.
- Bandpower values are log-transformed because spectral power is strongly right-skewed.
- Five bands across 18 channels produce a **90-dimensional feature vector** for each 4-second EEG window.

In [ ]:
PSD_DATA_ROOT = Path(
    "/content/drive/MyDrive/chbmit-seizure-detection/loso_psd_data"
)

PSD_DATA_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

FREQUENCY_BANDS = {
    "delta": (1, 4),
    "theta": (4, 8),
    "alpha": (8, 12),
    "beta": (12, 30),
    "gamma": (30, 50),
}


def extract_log_bandpower(X, sfreq=256):

    freqs, psd = welch(
        X,
        fs=sfreq,
        nperseg=256,
        axis=-1
    )

    feature_parts = []

    for low, high in FREQUENCY_BANDS.values():

        mask = (
            (freqs >= low)
            & (freqs < high)
        )

        band_power = trapezoid(
            psd[..., mask],
            x=freqs[mask],
            axis=-1
        )

        log_power = np.log10(
            band_power + 1e-12
        )

        feature_parts.append(
            log_power.astype(np.float32)
        )

    return np.concatenate(
        feature_parts,
        axis=1
    )

### 4.2 Generate Patient-Level PSD Features

- The same retained windows and labels from the raw EEG pipeline are reused so RAW and PSD are evaluated on identical samples.
- Each patient's raw windows are converted to 90 spectral features.
- Shape and finite-value checks verify the transformation before the features are saved as patient-specific `.npz` files.

In [ ]:
for subject in DEV_SUBJECTS:

    with np.load(
        LOSO_DATA_ROOT / f"{subject}.npz"
    ) as data:

        X = data["X"]
        y = data["y"]

    F = extract_log_bandpower(X)

    assert F.shape == (
        len(y),
        90
    )

    assert np.isfinite(F).all()

    np.savez_compressed(
        PSD_DATA_ROOT / f"{subject}.npz",
        X=F,
        y=y
    )

    print(
        subject,
        "|",
        F.shape
    )

    del X, y, F
    gc.collect()

chb01 | (1735, 90)
chb02 | (1077, 90)
chb03 | (1736, 90)
chb04 | (5919, 90)
chb05 | (1736, 90)
chb06 | (6766, 90)
chb07 | (4468, 90)
chb08 | (1736, 90)
chb09 | (7105, 90)
chb10 | (3539, 90)
chb11 | (1736, 90)
chb13 | (1736, 90)
chb14 | (1705, 90)
chb17 | (1736, 90)
chb19 | (1737, 90)
chb20 | (1706, 90)
chb22 | (1734, 90)
chb23 | (4364, 90)


### 4.3 Fold-Specific PSD Normalization

- PSD features are standardized **feature-by-feature** before training.
- As with the raw EEG pipeline, the mean and standard deviation are calculated using only the **17 training patients** in each LOSO fold.
- The same training-derived statistics are then applied to the held-out patient's features, preventing preprocessing leakage.

In [ ]:
def load_psd_subject(subject):

    with np.load(
        PSD_DATA_ROOT / f"{subject}.npz"
    ) as data:

        X = data["X"].astype(np.float32)
        y = data["y"].astype(np.float32)

    return X, y


def fit_psd_scaler(subjects):

    feature_sum = np.zeros(
        90,
        dtype=np.float64
    )

    feature_sq_sum = np.zeros(
        90,
        dtype=np.float64
    )

    count = 0

    for subject in subjects:

        X, _ = load_psd_subject(subject)

        feature_sum += X.sum(
            axis=0,
            dtype=np.float64
        )

        feature_sq_sum += np.square(
            X,
            dtype=np.float64
        ).sum(axis=0)

        count += len(X)

        del X

    mean = feature_sum / count

    variance = (
        feature_sq_sum / count
        - mean ** 2
    )

    std = np.sqrt(
        np.maximum(
            variance,
            1e-8
        )
    )

    return (
        mean.astype(np.float32),
        std.astype(np.float32)
    )


def build_psd_arrays(
    subjects,
    mean,
    std
):

    X_parts = []
    y_parts = []

    for subject in subjects:

        X, y = load_psd_subject(subject)

        X = (
            X - mean
        ) / (
            std + 1e-6
        )

        X_parts.append(
            X.astype(np.float32)
        )

        y_parts.append(y)

    return (
        np.concatenate(X_parts),
        np.concatenate(y_parts)
    )

### 4.4 PSD Multilayer Perceptron

- The 90-dimensional spectral feature vector is classified with a **multilayer perceptron (MLP)** rather than a convolutional model because the temporal waveform has already been reduced to fixed bandpower features.
- Two hidden layers reduce the representation from **90 → 64 → 32** features.
- **ReLU** provides nonlinear transformations, while **dropout (0.3)** is applied after each hidden layer to reduce overfitting.
- The final linear layer outputs one logit for binary seizure classification.

In [ ]:
class LOSOPSDMLP(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(90, 64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(32, 1)
        )

    def forward(self, x):

        return (
            self.network(x)
            .squeeze(1)
        )


print(LOSOPSDMLP())

LOSOPSDMLP(
  (network): Sequential(
    (0): Linear(in_features=90, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=32, out_features=1, bias=True)
  )
)


### 4.5 One-Epoch PSD LOSO Baseline

- The PSD model uses the **same 18 LOSO folds**, training on 17 patients and evaluating on the remaining patient.
- Training-only feature normalization and class weighting follow the same procedure used for the raw EEG model.
- The MLP uses the same **AdamW** optimizer settings and is trained for exactly **one epoch**.
- AUROC and AUPRC are calculated separately for each held-out patient, allowing direct paired comparison with the raw EEG model.

In [ ]:
def run_fixed_psd_fold(outer_subject):

    print("\n================================")
    print("OUTER PATIENT:", outer_subject)
    print("================================")

    train_subjects = [
        subject
        for subject in DEV_SUBJECTS
        if subject != outer_subject
    ]

    set_seed(SEED)

    mean, std = fit_psd_scaler(
        train_subjects
    )

    X_train, y_train = build_psd_arrays(
        train_subjects,
        mean,
        std
    )

    X_outer, y_outer = build_psd_arrays(
        [outer_subject],
        mean,
        std
    )

    train_loader = make_loader(
        X_train,
        y_train,
        shuffle=True
    )

    outer_loader = make_loader(
        X_outer,
        y_outer,
        shuffle=False
    )

    n_positive = (y_train == 1).sum()
    n_negative = (y_train == 0).sum()

    pos_weight = (
        n_negative / n_positive
    )

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(
            [pos_weight],
            dtype=torch.float32,
            device=device
        )
    )

    model = LOSOPSDMLP().to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-4
    )

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion
    )

    outer_labels, outer_probs = predict_probabilities(
        model,
        outer_loader
    )

    auroc = roc_auc_score(
        outer_labels,
        outer_probs
    )

    auprc = average_precision_score(
        outer_labels,
        outer_probs
    )

    result = {
        "subject": outer_subject,
        "epochs": FIXED_EPOCHS,
        "train_loss": float(train_loss),
        "positives": int(outer_labels.sum()),
        "windows": len(outer_labels),
        "prevalence": float(outer_labels.mean()),
        "auroc": float(auroc),
        "auprc": float(auprc),
    }

    print(f"Train loss: {train_loss:.4f}")
    print(f"OUTER AUROC: {auroc:.4f}")
    print(f"OUTER AUPRC: {auprc:.4f}")

    del (
        X_train,
        y_train,
        X_outer,
        y_outer,
        train_loader,
        outer_loader,
        model
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result

### 4.6 Running or Loading PSD LOSO Results

- The full 18-fold PSD LOSO experiment is loaded from saved results by default.
- Setting `RERUN_PSD_LOSO = True` intentionally recomputes all 18 folds.
- Results are saved after each completed fold so progress is preserved if execution is interrupted.
- The final summary reports the median patient-level AUROC and AUPRC across all 18 held-out patients.

In [ ]:
PSD_RESULTS_PATH = (
    "/content/drive/MyDrive/chbmit-seizure-detection/"
    "fixed_psd_loso_18patients.csv"
)

RERUN_PSD_LOSO = False


if RERUN_PSD_LOSO:

    fixed_psd_results = []

    for i, subject in enumerate(
        DEV_SUBJECTS,
        start=1
    ):

        print(
            f"\n######## PSD FOLD "
            f"{i}/{len(DEV_SUBJECTS)} ########"
        )

        fixed_psd_results.append(
            run_fixed_psd_fold(subject)
        )

        pd.DataFrame(
            fixed_psd_results
        ).to_csv(
            PSD_RESULTS_PATH,
            index=False
        )

    fixed_psd_df = pd.DataFrame(
        fixed_psd_results
    )

else:

    fixed_psd_df = pd.read_csv(
        PSD_RESULTS_PATH
    )


display(fixed_psd_df)

print("\nPSD LOSO SUMMARY")

print(
    "Median AUROC:",
    fixed_psd_df["auroc"].median()
)

print(
    "Median AUPRC:",
    fixed_psd_df["auprc"].median()
)

,subject,epochs,train_loss,positives,windows,prevalence,auroc,auprc
0,chb01,1,0.875211,17,1735,0.009798,0.987057,0.925062
1,chb02,1,0.875714,24,1077,0.022284,0.960391,0.230931
2,chb03,1,0.871264,30,1736,0.017281,0.979465,0.284493
3,chb04,1,0.866186,40,5919,0.006758,0.953942,0.161103
4,chb05,1,0.896276,57,1736,0.032834,0.956720,0.358324
5,chb06,1,0.751527,22,6766,0.003252,0.371482,0.002485
6,chb07,1,0.922946,46,4468,0.010295,0.994735,0.722708
7,chb08,1,0.858520,91,1736,0.052419,0.778149,0.364336
8,chb09,1,0.877707,54,7105,0.007600,0.994162,0.434123
9,chb10,1,0.870496,27,3539,0.007629,0.535276,0.008170



PSD LOSO SUMMARY
Median AUROC: 0.9585556082520215
Median AUPRC: 0.3992297902274373


## 5. Paired Patient-Level Statistical Analysis

RAW and PSD are evaluated on the **same 18 held-out patients**, so their performance measurements are paired.

- The **patient**, rather than the individual EEG window, is the statistical unit.
- RAW and PSD results are matched by patient before calculating performance differences.
- Differences are defined as **PSD − RAW**, so negative values favor the raw representation.
- Statistical significance is evaluated at **α = 0.05**.

### 5.1 Pair Patient-Level Results

In [ ]:
FINAL_STATS_PATH = (
    "/content/drive/MyDrive/chbmit-seizure-detection/"
    "raw_vs_psd_FINAL_18patients.csv"
)

raw = fixed_raw_df.copy()
psd = fixed_psd_df.copy()

# Verify the expected 18-patient paired experiment
assert len(raw) == 18
assert len(psd) == 18
assert raw["subject"].nunique() == 18
assert psd["subject"].nunique() == 18

paired = raw[
    ["subject", "auroc", "auprc"]
].merge(
    psd[
        ["subject", "auroc", "auprc"]
    ],
    on="subject",
    suffixes=("_raw", "_psd"),
    validate="one_to_one"
)

assert len(paired) == 18

paired["delta_auroc"] = (
    paired["auroc_psd"]
    - paired["auroc_raw"]
)

paired["delta_auprc"] = (
    paired["auprc_psd"]
    - paired["auprc_raw"]
)

display(paired)

,subject,auroc_raw,auprc_raw,auroc_psd,auprc_psd,delta_auroc,delta_auprc
0,chb01,0.996645,0.946488,0.987057,0.925062,-0.009587,-0.021426
1,chb02,0.954614,0.248967,0.960391,0.230931,0.005777,-0.018037
2,chb03,0.988550,0.645840,0.979465,0.284493,-0.009086,-0.361347
3,chb04,0.984508,0.251425,0.953942,0.161103,-0.030566,-0.090322
4,chb05,0.903848,0.804108,0.956720,0.358324,0.052872,-0.445784
5,chb06,0.741433,0.009442,0.371482,0.002485,-0.369952,-0.006957
6,chb07,0.995993,0.853418,0.994735,0.722708,-0.001259,-0.130710
7,chb08,0.880818,0.582970,0.778149,0.364336,-0.102669,-0.218633
8,chb09,0.997269,0.944946,0.994162,0.434123,-0.003107,-0.510822
9,chb10,0.980933,0.312493,0.535276,0.008170,-0.445657,-0.304323


### 5.2 Statistical Tests

Two paired nonparametric analyses are used:

- The **Wilcoxon signed-rank test** tests whether the paired RAW–PSD performance differences are systematically shifted away from zero without assuming normally distributed patient-level differences.
- An **exact paired sign-flip test** is used as a sensitivity analysis. Under the null hypothesis, the sign of each patient's RAW–PSD difference is exchangeable.
- With 18 patients, all **262,144 possible sign assignments** can be enumerated exactly rather than estimated through random sampling.
- Both tests are **two-sided**.

In [ ]:
ALPHA = 0.05


wilcox_auc = wilcoxon(
    paired["auroc_psd"],
    paired["auroc_raw"],
    alternative="two-sided"
)

wilcox_pr = wilcoxon(
    paired["auprc_psd"],
    paired["auprc_raw"],
    alternative="two-sided"
)


def exact_paired_permutation_test(differences):

    differences = np.asarray(
        differences,
        dtype=np.float64
    )

    observed = differences.mean()

    extreme = 0
    total = 0

    for signs in product(
        (-1.0, 1.0),
        repeat=len(differences)
    ):

        permuted_mean = np.mean(
            differences
            * np.asarray(signs)
        )

        if (
            abs(permuted_mean)
            >= abs(observed) - 1e-15
        ):
            extreme += 1

        total += 1

    p_value = extreme / total

    return (
        observed,
        p_value,
        total
    )


perm_auc_mean, perm_auc_p, n_permutations = (
    exact_paired_permutation_test(
        paired["delta_auroc"].values
    )
)

perm_pr_mean, perm_pr_p, _ = (
    exact_paired_permutation_test(
        paired["delta_auprc"].values
    )
)

### 5.3 Statistical Results

In [ ]:
print("RAW vs PSD — 18 paired held-out patients")

print("\nMedian performance")
print(
    f"RAW AUROC:  {paired['auroc_raw'].median():.6f}"
)
print(
    f"PSD AUROC:  {paired['auroc_psd'].median():.6f}"
)
print(
    f"RAW AUPRC:  {paired['auprc_raw'].median():.6f}"
)
print(
    f"PSD AUPRC:  {paired['auprc_psd'].median():.6f}"
)

print("\nPaired differences (PSD - RAW)")
print(
    f"Median AUROC delta: "
    f"{paired['delta_auroc'].median():.6f}"
)
print(
    f"Median AUPRC delta: "
    f"{paired['delta_auprc'].median():.6f}"
)
print(
    f"Mean AUROC delta: "
    f"{paired['delta_auroc'].mean():.6f}"
)
print(
    f"Mean AUPRC delta: "
    f"{paired['delta_auprc'].mean():.6f}"
)

print(
    "\nPSD better AUROC:",
    int((paired["delta_auroc"] > 0).sum()),
    "/ 18"
)

print(
    "PSD better AUPRC:",
    int((paired["delta_auprc"] > 0).sum()),
    "/ 18"
)

print("\nWilcoxon signed-rank")
print(
    f"AUROC: W = {wilcox_auc.statistic:.3f}, "
    f"p = {wilcox_auc.pvalue:.6f}"
)
print(
    f"AUPRC: W = {wilcox_pr.statistic:.3f}, "
    f"p = {wilcox_pr.pvalue:.6f}"
)

print("\nExact paired sign-flip test")
print(
    f"AUROC: mean delta = {perm_auc_mean:.6f}, "
    f"p = {perm_auc_p:.6f}"
)
print(
    f"AUPRC: mean delta = {perm_pr_mean:.6f}, "
    f"p = {perm_pr_p:.6f}"
)
print(
    f"Permutations evaluated: "
    f"{n_permutations:,}"
)

paired.to_csv(
    FINAL_STATS_PATH,
    index=False
)

RAW vs PSD — 18 paired held-out patients

Median performance
RAW AUROC:  0.981182
PSD AUROC:  0.958556
RAW AUPRC:  0.613816
PSD AUPRC:  0.399230

Paired differences (PSD - RAW)
Median AUROC delta: -0.001355
Median AUPRC delta: -0.012497
Mean AUROC delta: -0.033897
Mean AUPRC delta: -0.091882

PSD better AUROC: 7 / 18
PSD better AUPRC: 7 / 18

Wilcoxon signed-rank
AUROC: W = 66.000, p = 0.417114
AUPRC: W = 53.000, p = 0.167351

Exact paired sign-flip test
AUROC: mean delta = -0.033897, p = 0.381966
AUPRC: mean delta = -0.091882, p = 0.064415
Permutations evaluated: 262,144


## 6. Results

Across the 18 held-out patients, the raw EEG CNN achieved a median AUROC of **0.981** and a median AUPRC of **0.614**. The PSD MLP achieved a median AUROC of **0.959** and a median AUPRC of **0.399**.

The median paired difference (PSD − RAW) was **−0.0014 AUROC** and **−0.0125 AUPRC**. PSD outperformed RAW on 7 of 18 patients for both AUROC and AUPRC, indicating substantial patient-to-patient variability rather than uniform dominance of one representation.

Neither paired statistical test reached the α = 0.05 significance threshold. The Wilcoxon signed-rank test produced **p = 0.417** for AUROC and **p = 0.167** for AUPRC. The exact paired sign-flip test produced **p = 0.382** for AUROC and **p = 0.064** for AUPRC.

## 7. Interpretation

Under this original one-epoch LOSO baseline, raw EEG showed stronger descriptive performance than the spectral representation, particularly for AUPRC. However, the paired patient-level analysis did **not** establish a statistically significant difference between the two representations.

The patient-level results also show substantial heterogeneity: PSD outperformed RAW for several patients even though RAW had the stronger overall median performance. This suggests that representation quality may depend partly on patient-specific EEG characteristics.

Because both models were trained for only one epoch, these results should be interpreted as a broad patient-level baseline rather than as a comparison of fully optimized models.

## 8. Limitations

- Both models were trained for only **one epoch**, so the experiment does not represent optimized model performance.
- The CNN and MLP have different architectures because they operate on fundamentally different input representations.
- The experiment uses a limited subset of CHB-MIT recordings rather than every available recording.
- Although 18 paired patients provide a meaningful patient-level comparison, statistical power remains limited.
- Results are specific to the preprocessing, windowing, and model configurations used here and should not be interpreted as clinical validation.